In [1]:
import pyodbc
import pandas as pd


# --------------------------- CONEXIONES ---------------------------

def get_connection_origen():
    try:
        conn = pyodbc.connect(
            'DRIVER={ODBC Driver 17 for SQL Server};'
            'SERVER=LAPTOP-RSS8LMV8;'
            'DATABASE=proyecto_origen;'
            'UID=sa;'
            'PWD=@DDJJrrvv1303',
            autocommit=False
        )
        print("Conexión a origen exitosa")
        return conn
    except pyodbc.Error as e:
        print(e)
        return None


def get_connection_destino():
    try:
        conn = pyodbc.connect(
            'DRIVER={ODBC Driver 17 for SQL Server};'
            'SERVER=LAPTOP-RSS8LMV8;'
            'DATABASE=proyecto_destino;'
            'UID=sa;'
            'PWD=@DDJJrrvv1303',
            autocommit=False
        )
        print("Conexión a destino exitosa")
        return conn
    except pyodbc.Error as e:
        print(e)
        return None


# --------------------------- CREAR ESQUEMA ---------------------------

def create_datamart_schema(cursor):
    cursor.execute("""
    IF OBJECT_ID('FactAccidente','U')    IS NOT NULL DROP TABLE FactAccidente;
    IF OBJECT_ID('DimSeverity','U')      IS NOT NULL DROP TABLE DimSeverity;
    IF OBJECT_ID('DimFecha','U')         IS NOT NULL DROP TABLE DimFecha;
    IF OBJECT_ID('DimUbicacionGeo','U')  IS NOT NULL DROP TABLE DimUbicacionGeo;
    IF OBJECT_ID('DimLugar','U')         IS NOT NULL DROP TABLE DimLugar;
    IF OBJECT_ID('DimClima','U')         IS NOT NULL DROP TABLE DimClima;
    IF OBJECT_ID('DimCrossing','U')      IS NOT NULL DROP TABLE DimCrossing;
    IF OBJECT_ID('DimJunction','U')      IS NOT NULL DROP TABLE DimJunction;
    IF OBJECT_ID('DimStation','U')       IS NOT NULL DROP TABLE DimStation;
    IF OBJECT_ID('DimStop','U')          IS NOT NULL DROP TABLE DimStop;
    IF OBJECT_ID('DimTraffic','U')       IS NOT NULL DROP TABLE DimTraffic;
    IF OBJECT_ID('DimCivilTwilight','U') IS NOT NULL DROP TABLE DimCivilTwilight;

    CREATE TABLE DimSeverity (
        id       INT IDENTITY(1,1) PRIMARY KEY,
        severity INT
    );
    CREATE TABLE DimFecha (
        id            INT IDENTITY(1,1) PRIMARY KEY,
        start_time    VARCHAR(16),
        anio          INT,
        mes           INT,
        dia           INT,
        hora          INT,
        dia_semana    VARCHAR(20),
        es_fin_semana BIT
    );
    CREATE TABLE DimUbicacionGeo (
        id        INT IDENTITY(1,1) PRIMARY KEY,
        start_lat FLOAT,
        start_lng FLOAT
    );
    CREATE TABLE DimLugar (
        id       INT IDENTITY(1,1) PRIMARY KEY,
        street   VARCHAR(255),
        city     VARCHAR(100),
        county   VARCHAR(100),
        state    VARCHAR(50),
        zipcode  VARCHAR(20),
        timezone VARCHAR(50)
    );
    CREATE TABLE DimClima (
        id                INT IDENTITY(1,1) PRIMARY KEY,
        weather_condition VARCHAR(100)
    );
    CREATE TABLE DimCrossing (
        id       INT IDENTITY(1,1) PRIMARY KEY,
        crossing VARCHAR(10)
    );
    CREATE TABLE DimJunction (
        id       INT IDENTITY(1,1) PRIMARY KEY,
        junction VARCHAR(10)
    );
    CREATE TABLE DimStation (
        id      INT IDENTITY(1,1) PRIMARY KEY,
        station VARCHAR(10)
    );
    CREATE TABLE DimStop (
        id   INT IDENTITY(1,1) PRIMARY KEY,
        stop VARCHAR(10)
    );
    CREATE TABLE DimTraffic (
        id             INT IDENTITY(1,1) PRIMARY KEY,
        traffic_signal VARCHAR(10)
    );
    CREATE TABLE DimCivilTwilight (
        id             INT IDENTITY(1,1) PRIMARY KEY,
        civil_twilight VARCHAR(20)
    );
    CREATE TABLE FactAccidente (
        id               VARCHAR(100) PRIMARY KEY,
        severityID       INT NOT NULL,
        fechaID          INT NOT NULL,
        ubicacionGeoID   INT NOT NULL,
        lugarID          INT NOT NULL,
        climaID          INT NOT NULL,
        crossingID       INT NOT NULL,
        junctionID       INT NOT NULL,
        stationID        INT NOT NULL,
        stopID           INT NOT NULL,
        trafficID        INT NOT NULL,
        civilTwilightID  INT NOT NULL,
        duracion_minutos INT,
        distance         FLOAT,
        temperature      FLOAT,
        wind_chill       FLOAT,
        humidity         FLOAT,
        pressure         FLOAT,
        visibility       FLOAT,
        wind_speed       FLOAT,
        precipitation    FLOAT,
        FOREIGN KEY (severityID)      REFERENCES DimSeverity(id),
        FOREIGN KEY (fechaID)         REFERENCES DimFecha(id),
        FOREIGN KEY (ubicacionGeoID)  REFERENCES DimUbicacionGeo(id),
        FOREIGN KEY (lugarID)         REFERENCES DimLugar(id),
        FOREIGN KEY (climaID)         REFERENCES DimClima(id),
        FOREIGN KEY (crossingID)      REFERENCES DimCrossing(id),
        FOREIGN KEY (junctionID)      REFERENCES DimJunction(id),
        FOREIGN KEY (stationID)       REFERENCES DimStation(id),
        FOREIGN KEY (stopID)          REFERENCES DimStop(id),
        FOREIGN KEY (trafficID)       REFERENCES DimTraffic(id),
        FOREIGN KEY (civilTwilightID) REFERENCES DimCivilTwilight(id)
    );
    """)
    cursor.commit()
    print("Esquema creado")


def create_indexes(cursor):
    cursor.execute("""
    CREATE INDEX IX_Fact_severityID     ON FactAccidente(severityID);
    CREATE INDEX IX_Fact_fechaID        ON FactAccidente(fechaID);
    CREATE INDEX IX_Fact_lugarID        ON FactAccidente(lugarID);
    CREATE INDEX IX_Fact_climaID        ON FactAccidente(climaID);
    CREATE INDEX IX_Fact_ubicacionGeoID ON FactAccidente(ubicacionGeoID);
    CREATE INDEX IX_DimLugar_city_state ON DimLugar(city, state);
    CREATE INDEX IX_DimFecha_time       ON DimFecha(start_time);
    CREATE INDEX IX_DimFecha_anio_mes   ON DimFecha(anio, mes);
    CREATE INDEX IX_DimFecha_hora       ON DimFecha(hora);
    """)
    cursor.commit()
    print("Índices creados")


# --------------------------- LIMPIAR DW ---------------------------

def clean_datamart(cursor):
    print("Limpiando DataMart...")
    cursor.execute("DELETE FROM FactAccidente")
    cursor.execute("DELETE FROM DimSeverity")
    cursor.execute("DELETE FROM DimFecha")
    cursor.execute("DELETE FROM DimUbicacionGeo")
    cursor.execute("DELETE FROM DimLugar")
    cursor.execute("DELETE FROM DimClima")
    cursor.execute("DELETE FROM DimCrossing")
    cursor.execute("DELETE FROM DimJunction")
    cursor.execute("DELETE FROM DimStation")
    cursor.execute("DELETE FROM DimStop")
    cursor.execute("DELETE FROM DimTraffic")
    cursor.execute("DELETE FROM DimCivilTwilight")
    cursor.commit()
    print("DataMart limpio")

# --------------------------- HELPERS ---------------------------
 
def to_float(val):
    """Convierte un valor a float o None si no es parseable."""
    try:
        f = float(val)
        return None if pd.isna(f) else f
    except (ValueError, TypeError):
        return None
 
 
def to_int(val):
    """Convierte un valor a int o None si no es parseable."""
    try:
        f = float(val)
        return None if pd.isna(f) else int(f)
    except (ValueError, TypeError):
        return None
    
def _insert_in_chunks(dest_cursor, sql, rows, chunk_size=500_000, label=""):
    dest_cursor.fast_executemany = True
    total = 0
    for i in range(0, len(rows), chunk_size):
        batch = rows[i: i + chunk_size]
        dest_cursor.executemany(sql, batch)
        dest_cursor.commit()
        total += len(batch)
        if label:
            print(f"  {total:,} / {len(rows):,} {label}...")
    return total
# --------------------------- DIMENSIONES ---------------------------

def load_dim_severity(source_conn, dest_cursor):
    print("Cargando DimSeverity...")
    df = pd.read_sql("SELECT DISTINCT Severity AS severity FROM accidents_raw WHERE Severity IS NOT NULL", source_conn)
    df['severity'] = pd.to_numeric(df['severity'], errors='coerce').dropna().astype(int)
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        "INSERT INTO DimSeverity (severity) VALUES (?)",
        [[int(v)] for v in df['severity']]
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en DimSeverity")

def load_dim_fecha(source_conn, dest_cursor):
    print("Cargando DimFecha...")
    df = pd.read_sql("""
        SELECT CONVERT(VARCHAR(16), Start_Time, 120) AS start_time
        FROM accidents_raw
        WHERE Start_Time IS NOT NULL
        GROUP BY CONVERT(VARCHAR(16), Start_Time, 120)
    """, source_conn)

    df['start_time'] = df['start_time'].fillna('')
    dt = pd.to_datetime(df['start_time'], format='%Y-%m-%d %H:%M', errors='coerce')
    df['anio']          = dt.dt.year.fillna(0).astype(int)
    df['mes']           = dt.dt.month.fillna(0).astype(int)
    df['dia']           = dt.dt.day.fillna(0).astype(int)
    df['hora']          = dt.dt.hour.fillna(0).astype(int)
    df['dia_semana']    = dt.dt.day_name().fillna('')
    df['es_fin_semana'] = dt.dt.dayofweek.isin([5, 6]).astype(int)

    rows = df[['start_time', 'anio', 'mes', 'dia', 'hora',
               'dia_semana', 'es_fin_semana']].values.tolist()

    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        "INSERT INTO DimFecha (start_time, anio, mes, dia, hora, dia_semana, es_fin_semana) VALUES (?,?,?,?,?,?,?)",
        rows
    )
    dest_cursor.commit()
    print(f"  {len(rows):,} registros en DimFecha")


def load_dim_ubicacion_geo(source_conn, dest_cursor):
    print("Cargando DimUbicacionGeo...")
    cursor = source_conn.cursor()
    cursor.execute("""
        SELECT
            DISTINCT
            CAST(Start_Lat AS FLOAT) AS start_lat,
            CAST(Start_Lng AS FLOAT) AS start_lng
        FROM accidents_raw
        WHERE Start_Lat IS NOT NULL
          AND Start_Lng IS NOT NULL
    """)
    rows = [list(r) for r in cursor.fetchall()]
    cursor.close()
    print(f"  Coordenadas únicas: {len(rows):,}")

    # ✅ chunks más pequeños para no saturar el driver con FLOATs
    _insert_in_chunks(dest_cursor,
        "INSERT INTO DimUbicacionGeo (start_lat, start_lng) VALUES (?,?)",
        rows, chunk_size=100_000, label="coordenadas")
    print(f"  {len(rows):,} registros en DimUbicacionGeo")


def load_dim_lugar(source_conn, dest_cursor):
    print("Cargando DimLugar...")
    df = pd.read_sql("""
        SELECT DISTINCT
            ISNULL(Street,  '')   AS street,
            ISNULL(City,    '')   AS city,
            ISNULL(County,  '')   AS county,
            ISNULL(State,   '')   AS state,
            ISNULL(Zipcode, '')   AS zipcode,
            ISNULL(Timezone,'')   AS timezone
        FROM accidents_raw
        WHERE City IS NOT NULL AND State IS NOT NULL
    """, source_conn)
    df = df.fillna('')
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        "INSERT INTO DimLugar (street, city, county, state, zipcode, timezone) VALUES (?,?,?,?,?,?)",
        df.values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en DimLugar")


def load_dim_clima(source_conn, dest_cursor):
    print("Cargando DimClima...")
    df = pd.read_sql("""
        SELECT DISTINCT Weather_Condition AS weather_condition
        FROM accidents_raw
        WHERE Weather_Condition IS NOT NULL
    """, source_conn)
    df = df.fillna('')
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        "INSERT INTO DimClima (weather_condition) VALUES (?)",
        df.values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en DimClima")


def _load_single_col_dim(source_conn, dest_cursor, col_src, table, col_dest):
    """Genérico para dimensiones de una sola columna booleana/categórica."""
    print(f"Cargando {table}...")
    df = pd.read_sql(f"""
        SELECT DISTINCT CAST({col_src} AS VARCHAR(10)) AS val
        FROM accidents_raw
        WHERE {col_src} IS NOT NULL
    """, source_conn)
    df = df.fillna('')
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        f"INSERT INTO {table} ({col_dest}) VALUES (?)",
        df.values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en {table}")


def load_dim_civil_twilight(source_conn, dest_cursor):
    print("Cargando DimCivilTwilight...")
    df = pd.read_sql("""
        SELECT DISTINCT Civil_Twilight AS civil_twilight
        FROM accidents_raw
        WHERE Civil_Twilight IS NOT NULL
    """, source_conn)
    df = df.fillna('')
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        "INSERT INTO DimCivilTwilight (civil_twilight) VALUES (?)",
        df.values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en DimCivilTwilight")


# --------------------------- MAPEOS ---------------------------

def build_maps(dest_conn):
    print("Construyendo mapeos en RAM...")

    df = pd.read_sql("SELECT id, severity FROM DimSeverity", dest_conn)
    # La clave del mapa debe ser str porque del origen viene como VARCHAR
    map_severity = {str(int(r.severity)): r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, start_time FROM DimFecha", dest_conn).fillna('')
    map_fecha = {r.start_time: r.id for r in df.itertuples(index=False)}
    
    df = pd.read_sql("SELECT id, start_lat, start_lng FROM DimUbicacionGeo",dest_conn).dropna()
    df['start_lat'] = df['start_lat'].round(6)
    df['start_lng'] = df['start_lng'].round(6)
    map_geo = {(lat, lng): id_ for lat, lng, id_ in zip(df['start_lat'], df['start_lng'], df['id'])}

    df = pd.read_sql("SELECT id, street, city, county, state FROM DimLugar", dest_conn).fillna('')
    map_lugar = {(r.street, r.city, r.county, r.state): r.id
                 for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, weather_condition FROM DimClima", dest_conn).fillna('')
    map_clima = {r.weather_condition: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, crossing FROM DimCrossing", dest_conn).fillna('')
    map_crossing = {r.crossing: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, junction FROM DimJunction", dest_conn).fillna('')
    map_junction = {r.junction: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, station FROM DimStation", dest_conn).fillna('')
    map_station = {r.station: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, stop FROM DimStop", dest_conn).fillna('')
    map_stop = {r.stop: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, traffic_signal FROM DimTraffic", dest_conn).fillna('')
    map_traffic = {r.traffic_signal: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, civil_twilight FROM DimCivilTwilight", dest_conn).fillna('')
    map_twilight = {r.civil_twilight: r.id for r in df.itertuples(index=False)}

    print("  Mapeos listos")
    return (map_severity, map_fecha, map_geo, map_lugar, map_clima,
            map_crossing, map_junction, map_station, map_stop,
            map_traffic, map_twilight)


# --------------------------- FACT TABLE ---------------------------

def load_fact(source_conn, dest_cursor,
              map_severity, map_fecha, map_geo, map_lugar, map_clima,
              map_crossing, map_junction, map_station, map_stop,
              map_traffic, map_twilight):

    print("Cargando FactAccidente...")

    # CORRECCIÓN: las métricas se leen como FLOAT directamente (no como VARCHAR)
    query = """
        SELECT
            ID,
            CAST(Severity AS VARCHAR(10))             AS severity,
            CONVERT(VARCHAR(16), Start_Time, 120)     AS start_time,
            CAST(Start_Lat AS FLOAT)                  AS start_lat,
            CAST(Start_Lng AS FLOAT)                  AS start_lng,
            ISNULL(Street,  '')                       AS Street,
            ISNULL(City,    '')                       AS City,
            ISNULL(County,  '')                       AS County,
            ISNULL(State,   '')                       AS State,
            Weather_Condition,
            CAST(Crossing       AS VARCHAR(10))       AS crossing,
            CAST(Junction       AS VARCHAR(10))       AS junction,
            CAST(Station        AS VARCHAR(10))       AS station,
            CAST(Stop           AS VARCHAR(10))       AS stop,
            CAST(Traffic_Signal AS VARCHAR(10))       AS traffic_signal,
            Civil_Twilight,
            DATEDIFF(
                MINUTE,
                TRY_CAST(Start_Time AS DATETIME),
                TRY_CAST(End_Time   AS DATETIME)
            )                                         AS duracion_minutos,
            CAST([Distance(mi)]      AS FLOAT)        AS distance,
            CAST([Temperature(F)]    AS FLOAT)        AS temperature,
            CAST([Wind_Chill(F)]     AS FLOAT)        AS wind_chill,
            CAST([Humidity(%)]       AS FLOAT)        AS humidity,
            CAST([Pressure(in)]      AS FLOAT)        AS pressure,
            CAST([Visibility(mi)]    AS FLOAT)        AS visibility,
            CAST([Wind_Speed(mph)]   AS FLOAT)        AS wind_speed,
            CAST([Precipitation(in)] AS FLOAT)        AS precipitation
        FROM accidents_raw
    """

    CHUNK_SIZE = 500_000
    total   = 0
    skipped = 0
    dest_cursor.fast_executemany = True

    for chunk in pd.read_sql(query, source_conn, chunksize=CHUNK_SIZE):

        chunk = chunk.dropna(subset=['start_time', 'severity', 'Weather_Condition'])
        chunk['start_time'] = chunk['start_time'].fillna('')
        chunk[['Street', 'City', 'County', 'State',
               'crossing', 'junction', 'station', 'stop',
               'traffic_signal', 'Civil_Twilight']] = \
            chunk[['Street', 'City', 'County', 'State',
                   'crossing', 'junction', 'station', 'stop',
                   'traffic_signal', 'Civil_Twilight']].fillna('')

        # --- Resolver FK de severidad (la clave del mapa es str del int) ---
        chunk['_sev_key'] = chunk['severity'].apply(
            lambda v: str(int(float(v))) if v not in ('', None) else ''
        )
        chunk['severityID'] = chunk['_sev_key'].map(map_severity)

        chunk['fechaID'] = chunk['start_time'].map(map_fecha)

        # --- Geo: redondear para que coincida con las claves del mapa ---
        chunk['_lat_r'] = chunk['start_lat'].apply(
            lambda v: round(float(v), 6) if pd.notna(v) else None
        )
        chunk['_lng_r'] = chunk['start_lng'].apply(
            lambda v: round(float(v), 6) if pd.notna(v) else None
        )
        chunk['ubicacionGeoID'] = list(zip(chunk['_lat_r'], chunk['_lng_r']))
        chunk['ubicacionGeoID'] = chunk['ubicacionGeoID'].map(map_geo)

        chunk['lugarID'] = list(zip(
            chunk['Street'], chunk['City'], chunk['County'], chunk['State']
        ))
        chunk['lugarID'] = chunk['lugarID'].map(map_lugar)

        chunk['climaID']         = chunk['Weather_Condition'].map(map_clima)
        chunk['crossingID']      = chunk['crossing'].map(map_crossing)
        chunk['junctionID']      = chunk['junction'].map(map_junction)
        chunk['stationID']       = chunk['station'].map(map_station)
        chunk['stopID']          = chunk['stop'].map(map_stop)
        chunk['trafficID']       = chunk['traffic_signal'].map(map_traffic)
        chunk['civilTwilightID'] = chunk['Civil_Twilight'].map(map_twilight)

        fk_cols = ['severityID', 'fechaID', 'ubicacionGeoID', 'lugarID',
                   'climaID', 'crossingID', 'junctionID', 'stationID',
                   'stopID', 'trafficID', 'civilTwilightID']
        before   = len(chunk)
        chunk    = chunk.dropna(subset=fk_cols)
        skipped += before - len(chunk)

        # --- Construir lista de tuplas con tipos correctos ---
        data = []
        for r in chunk.itertuples(index=False):
            data.append((
                str(r.ID),
                int(r.severityID),
                int(r.fechaID),
                int(r.ubicacionGeoID),
                int(r.lugarID),
                int(r.climaID),
                int(r.crossingID),
                int(r.junctionID),
                int(r.stationID),
                int(r.stopID),
                int(r.trafficID),
                int(r.civilTwilightID),
                to_int(r.duracion_minutos),   # INT o None
                to_float(r.distance),          # FLOAT o None
                to_float(r.temperature),
                to_float(r.wind_chill),
                to_float(r.humidity),
                to_float(r.pressure),
                to_float(r.visibility),
                to_float(r.wind_speed),
                to_float(r.precipitation),
            ))

        dest_cursor.executemany("""
            INSERT INTO FactAccidente (
                id, severityID, fechaID, ubicacionGeoID, lugarID,
                climaID, crossingID, junctionID, stationID, stopID,
                trafficID, civilTwilightID,
                duracion_minutos, distance, temperature, wind_chill, humidity,
                pressure, visibility, wind_speed, precipitation
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, data)
        dest_cursor.commit()

        total += len(data)
        print(f"  {total:,} filas insertadas  ({skipped:,} omitidas por FK nula)...")

    print(f"\n  FactAccidente completa: {total:,} registros  |  omitidos: {skipped:,}")


# --------------------------- MAIN ---------------------------

if __name__ == "__main__":

    source_conn = get_connection_origen()
    dest_conn   = get_connection_destino()

    if not source_conn or not dest_conn:
        exit(1)

    dest_cursor = dest_conn.cursor()

    create_datamart_schema(dest_cursor)
    clean_datamart(dest_cursor)

    load_dim_severity(source_conn, dest_cursor)
    load_dim_fecha(source_conn, dest_cursor)
    load_dim_ubicacion_geo(source_conn, dest_cursor)
    load_dim_lugar(source_conn, dest_cursor)
    load_dim_clima(source_conn, dest_cursor)
    _load_single_col_dim(source_conn, dest_cursor, 'Crossing',       'DimCrossing', 'crossing')
    _load_single_col_dim(source_conn, dest_cursor, 'Junction',       'DimJunction', 'junction')
    _load_single_col_dim(source_conn, dest_cursor, 'Station',        'DimStation',  'station')
    _load_single_col_dim(source_conn, dest_cursor, 'Stop',           'DimStop',     'stop')
    _load_single_col_dim(source_conn, dest_cursor, 'Traffic_Signal', 'DimTraffic',  'traffic_signal')
    load_dim_civil_twilight(source_conn, dest_cursor)

    create_indexes(dest_cursor)

    maps = build_maps(dest_conn)

    load_fact(source_conn, dest_cursor, *maps)

    source_conn.close()
    dest_conn.close()
    print("\nETL completado exitosamente")

Conexión a origen exitosa
Conexión a destino exitosa
Esquema creado
Limpiando DataMart...
DataMart limpio
Cargando DimSeverity...


C:\Users\Diego\AppData\Local\Temp\ipykernel_1964\1367667972.py:222: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT DISTINCT Severity AS severity FROM accidents_raw WHERE Severity IS NOT NULL", source_conn)


  4 registros en DimSeverity
Cargando DimFecha...


C:\Users\Diego\AppData\Local\Temp\ipykernel_1964\1367667972.py:234: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


  2,361,526 registros en DimFecha
Cargando DimUbicacionGeo...
  Coordenadas únicas: 3,005,430
  100,000 / 3,005,430 coordenadas...
  200,000 / 3,005,430 coordenadas...
  300,000 / 3,005,430 coordenadas...
  400,000 / 3,005,430 coordenadas...
  500,000 / 3,005,430 coordenadas...
  600,000 / 3,005,430 coordenadas...
  700,000 / 3,005,430 coordenadas...
  800,000 / 3,005,430 coordenadas...
  900,000 / 3,005,430 coordenadas...
  1,000,000 / 3,005,430 coordenadas...
  1,100,000 / 3,005,430 coordenadas...
  1,200,000 / 3,005,430 coordenadas...
  1,300,000 / 3,005,430 coordenadas...
  1,400,000 / 3,005,430 coordenadas...
  1,500,000 / 3,005,430 coordenadas...
  1,600,000 / 3,005,430 coordenadas...
  1,700,000 / 3,005,430 coordenadas...
  1,800,000 / 3,005,430 coordenadas...
  1,900,000 / 3,005,430 coordenadas...
  2,000,000 / 3,005,430 coordenadas...
  2,100,000 / 3,005,430 coordenadas...
  2,200,000 / 3,005,430 coordenadas...
  2,300,000 / 3,005,430 coordenadas...
  2,400,000 / 3,005,430 coo

...

  2 registros en DimCrossing
Cargando DimJunction...
  2 registros en DimJunction
Cargando DimStation...
  2 registros en DimStation
Cargando DimStop...
  2 registros en DimStop
Cargando DimTraffic...
  2 registros en DimTraffic
Cargando DimCivilTwilight...


C:\Users\Diego\AppData\Local\Temp\ipykernel_1964\1367667972.py:345: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


  2 registros en DimCivilTwilight
Índices creados
Construyendo mapeos en RAM...


C:\Users\Diego\AppData\Local\Temp\ipykernel_1964\1367667972.py:365: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT id, severity FROM DimSeverity", dest_conn)
C:\Users\Diego\AppData\Local\Temp\ipykernel_1964\1367667972.py:369: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT id, start_time FROM DimFecha", dest_conn).fillna('')
C:\Users\Diego\AppData\Local\Temp\ipykernel_1964\1367667972.py:372: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT id, start_lat, s

  Mapeos listos
Cargando FactAccidente...
  489,767 filas insertadas  (48 omitidas por FK nula)...
  980,194 filas insertadas  (1,595 omitidas por FK nula)...
  1,470,986 filas insertadas  (1,620 omitidas por FK nula)...
  1,962,689 filas insertadas  (1,642 omitidas por FK nula)...
  2,455,368 filas insertadas  (1,660 omitidas por FK nula)...
  2,943,115 filas insertadas  (1,685 omitidas por FK nula)...
  3,430,878 filas insertadas  (1,753 omitidas por FK nula)...
  3,916,261 filas insertadas  (4,516 omitidas por FK nula)...
  4,400,613 filas insertadas  (8,374 omitidas por FK nula)...
  4,885,083 filas insertadas  (12,234 omitidas por FK nula)...
  5,369,724 filas insertadas  (15,916 omitidas por FK nula)...
  5,856,197 filas insertadas  (18,538 omitidas por FK nula)...
  6,342,494 filas insertadas  (20,685 omitidas por FK nula)...
  6,825,430 filas insertadas  (21,095 omitidas por FK nula)...
  7,312,347 filas insertadas  (21,159 omitidas por FK nula)...
  7,533,737 filas insertadas 